[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/36_int8_quantization_solution.ipynb)

# 🔴 Solution: INT8 Quantized Linear

*Inference & Decoding · Hard*

Reference implementation. Try it yourself in `36_int8_quantization.ipynb` first.

---
Implement an **INT8-quantized linear layer** using symmetric per-channel
quantization.

$$s_o = \frac{\max_i |W_{oi}|}{127}, \qquad
  W^{q}_{oi} = \mathrm{clip}\big(\mathrm{round}(W_{oi}/s_o),\ -128,\ 127\big)$$

and at inference $\hat{W} = W^{q} \cdot s$, then $y = x\hat{W}^\top + b$.

### Signature
```python
class Int8Linear(nnx.Module):
    def __init__(self, weight, bias=None): ...   # weight: (out, in)
    def __call__(self, x): ...
```

### Rules
- **Symmetric**: zero maps to zero, so there is no zero-point
- **Per output channel**: one scale per row of `weight`, shape `(out, 1)`
- Quantize with `round`, then clip to `[-128, 127]`, then cast to `int8`
- Guard the division with `1e-10` so an all-zero row does not produce `NaN`
- Store `weight_int8` and `scale` as `nnx.Variable` (buffers); `bias` stays an
  `nnx.Param`

### Per-tensor vs per-channel
A single scale for the whole matrix is per-*tensor* quantization. It is cheaper
but fragile: one output channel with an unusually large weight sets the scale
for every channel, and all the small ones collapse into a handful of integer
levels. Per-channel gives each row its own scale, costs one float per row, and
is what makes INT8 weight quantization essentially lossless in practice.

### What actually breaks in LLMs
Weight quantization is the easy half. **Activation** quantization is where INT8
falls over, because transformer activations develop systematic outlier
channels — a few dimensions with magnitudes 100× the rest, appearing past
roughly 6.7B parameters. One outlier sets the scale for the whole tensor and
destroys the rest. That observation is exactly what LLM.int8() addresses, by
keeping outlier channels in fp16 and quantizing only the well-behaved ones.

### Why dequantize at all
This implementation stores int8 and computes in float — the win is **memory**
and bandwidth (4× smaller weights), not arithmetic. True int8 matmul with int32
accumulation needs hardware support and a fused kernel; the dequantize-then-
matmul form is what you write when you are showing you understand the numerics.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class Int8Linear(nnx.Module):
    def __init__(self, weight, bias=None):
        # One scale per OUTPUT channel: reduce over the input axis.
        scale = jnp.max(jnp.abs(weight), axis=1, keepdims=True) / 127.0

        q = jnp.round(weight / (scale + 1e-10))
        q = jnp.clip(q, -128, 127).astype(jnp.int8)

        # Buffers, not parameters — nnx.Variable keeps them out of nnx.Param
        # so an optimizer never tries to train them.
        self.weight_int8 = nnx.Variable(q)
        self.scale = nnx.Variable(scale)
        self.bias = nnx.Param(bias) if bias is not None else None

    def __call__(self, x):
        # Dequantize, then matmul in float. The saving is memory, not flops.
        w = self.weight_int8[...].astype(x.dtype) * self.scale[...]
        out = x @ w.T
        if self.bias is not None:
            out = out + self.bias[...]
        return out

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

w = jax.random.normal(jax.random.key(0), (16, 32))
layer = Int8Linear(w)

x = jax.random.normal(jax.random.key(1), (4, 32))
exact = x @ w.T
approx = layer(x)

print("int8 dtype :", layer.weight_int8[...].dtype)
print("scale shape:", layer.scale[...].shape, "(one per output channel)")
print("rel error  :", float(jnp.abs(approx - exact).max() / jnp.abs(exact).max()))
print("memory     : 4x smaller weights (int8 vs float32)")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("int8_quantization")